# Cleanup and load data


In [ ]:
# ── Step 0: Cleanup ───────────────────────────────────────────────────────────
import os
import shutil

for folder in ['asl_alphabet_train', 'asl_alphabet_test', 'asl_combined_5class']:
    if os.path.exists(folder):
        shutil.rmtree(folder)
        print(f"Cleaned up old {folder}")

# ── Step 1: Unzip Kaggle dataset ──────────────────────────────────────────────
!unzip -q 'archive (1).zip'
#!mv asl_dataset__ asl_dataset
!unzip -q 'webcam_training_data.zip'

!echo "Unzip done."

Cleaned up old asl_alphabet_train
Cleaned up old asl_alphabet_test
Unzip done.


# April 7 personal/kaggle datset combination training A,M,N,O,T.

In [ ]:


# ── Step 2: Upload your webcam_training_data folder to Colab ──────────────────
# You should see webcam_training_data/A/, webcam_training_data/M/, etc.
# If you uploaded a zip, uncomment the next line:
# !unzip -q webcam_training_data.zip

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import json
import os
import shutil
import random
import gc
import numpy as np

# ── Config ────────────────────────────────────────────────────────────────────
IMAGE_SIZE       = (160, 160)
BATCH_SIZE       = 32
EPOCHS_FROZEN    = 40
EPOCHS_FINETUNE  = 20
KAGGLE_DIR       = 'asl_alphabet_train/asl_alphabet_train'
WEBCAM_DIR       = 'webcam_training_data'
COMBINED_DIR     = 'asl_combined_5class'
TARGET_CLASSES   = ['A', 'M', 'N', 'O', 'T']
MODEL_NAME       = 'asl_model_5class.keras'
CLASSES_NAME     = 'class_names_5class.json'
# ─────────────────────────────────────────────────────────────────────────────

if not os.path.exists(KAGGLE_DIR):
    print("Expected directory not found. Available directories:")
    os.system('find . -maxdepth 4 -type d')
    raise Exception("Update KAGGLE_DIR to match the correct path shown above")

print(f"Kaggle dataset found at: {KAGGLE_DIR}")

if not os.path.exists(WEBCAM_DIR):
    print(f"\nWARNING: '{WEBCAM_DIR}' not found!")
    print("Upload your webcam_training_data folder to Colab before running.")
    print("Continuing with Kaggle data only...\n")

# ── Build combined dataset with target classes ────────────────────────────────
if os.path.exists(COMBINED_DIR):
    shutil.rmtree(COMBINED_DIR)

print(f"\nBuilding 5-class dataset for: {TARGET_CLASSES}")

for cls in TARGET_CLASSES:
    dst_dir = os.path.join(COMBINED_DIR, cls)
    os.makedirs(dst_dir, exist_ok=True)

    # Copy ALL Kaggle images for this class (no cap — use everything)
    kaggle_cls = os.path.join(KAGGLE_DIR, cls)
    kaggle_count = 0
    if os.path.exists(kaggle_cls):
        images = [f for f in os.listdir(kaggle_cls) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        for img in images:
            shutil.copy(os.path.join(kaggle_cls, img), os.path.join(dst_dir, f"kaggle_{img}"))
        kaggle_count = len(images)

    # Copy ALL webcam images for this class (added on top)
    webcam_cls = os.path.join(WEBCAM_DIR, cls)
    webcam_count = 0
    if os.path.exists(webcam_cls):
        images = [f for f in os.listdir(webcam_cls) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        for img in images:
            shutil.copy(os.path.join(webcam_cls, img), os.path.join(dst_dir, f"webcam_{img}"))
        webcam_count = len(images)

    total = kaggle_count + webcam_count
    print(f"  {cls}: {kaggle_count} Kaggle + {webcam_count} webcam = {total} total")

gc.collect()

# ── Step 3: Load datasets ────────────────────────────────────────────────────
full_dataset = tf.keras.utils.image_dataset_from_directory(
    COMBINED_DIR,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    validation_split=0.2,
    subset='both',
    seed=42
)

train_dataset, validation_dataset = full_dataset

class_names = train_dataset.class_names
NUM_CLASSES = len(class_names)
print(f"\nClasses ({NUM_CLASSES}): {class_names}")

with open(CLASSES_NAME, 'w') as f:
    json.dump(class_names, f)
print(f"Saved {CLASSES_NAME}")

# ── Step 4: Pipeline ─────────────────────────────────────────────────────────
train_dataset = (
    train_dataset
    .shuffle(500)
    .prefetch(tf.data.AUTOTUNE)
)

validation_dataset = (
    validation_dataset
    .prefetch(tf.data.AUTOTUNE)
)

# ── Step 5: Augmentation ─────────────────────────────────────────────────────
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.10),       # Lower rotation — rotating T can look like N
    layers.RandomZoom(0.15),
    layers.RandomBrightness(0.3),
    layers.RandomContrast(0.3),
    layers.RandomTranslation(0.1, 0.1),
], name="augmentation")

# ── Step 6: Build model ──────────────────────────────────────────────────────
base_model = keras.applications.MobileNetV2(
    input_shape=IMAGE_SIZE + (3,),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

inputs = keras.Input(shape=IMAGE_SIZE + (3,))
x = data_augmentation(inputs)
x = keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(512, activation='relu')(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = keras.Model(inputs, outputs)

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ── Step 7: Phase 1 — Frozen backbone ────────────────────────────────────────
print("\n=== Phase 1: Training (backbone frozen) ===")
history = model.fit(
    train_dataset,
    epochs=EPOCHS_FROZEN,
    validation_data=validation_dataset,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=7, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3, verbose=1),
        keras.callbacks.ModelCheckpoint(
            MODEL_NAME,
            save_best_only=True,
            monitor='val_accuracy',
            verbose=1
        )
    ],
    verbose=1
)

results = model.evaluate(validation_dataset)
print(f"\nPhase 1 Val Loss: {results[0]:.4f} | Val Accuracy: {results[1]:.4f}")

# ── Step 8: Phase 2 — Fine-tune backbone ─────────────────────────────────────
print("\n=== Phase 2: Fine-tuning top 30 layers ===")
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

trainable_count = sum(1 for l in base_model.layers if l.trainable)
frozen_count = sum(1 for l in base_model.layers if not l.trainable)
print(f"Backbone layers — trainable: {trainable_count}, frozen: {frozen_count}")

model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_finetune = model.fit(
    train_dataset,
    epochs=EPOCHS_FINETUNE,
    validation_data=validation_dataset,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=2, verbose=1),
        keras.callbacks.ModelCheckpoint(
            MODEL_NAME,
            save_best_only=True,
            monitor='val_accuracy',
            verbose=1
        )
    ],
    verbose=1
)

# ── Step 9: Final evaluation ─────────────────────────────────────────────────
results = model.evaluate(validation_dataset)
print(f"\nFinal Val Loss: {results[0]:.4f} | Final Val Accuracy: {results[1]:.4f}")

# ── Step 10: Confusion matrix ────────────────────────────────────────────────
print("\n=== Confusion Matrix ===")

# Collect all predictions and true labels from validation set
all_predictions = []
all_labels = []

for images, labels in validation_dataset:
    preds = model.predict(images, verbose=0)
    all_predictions.extend(np.argmax(preds, axis=1))
    all_labels.extend(np.argmax(labels.numpy(), axis=1))

all_predictions = np.array(all_predictions)
all_labels = np.array(all_labels)

# Print text confusion matrix
print(f"\n{'':>8}", end="")
for name in class_names:
    print(f"{name:>6}", end="")
print("  <- Predicted")
print("-" * (8 + 6 * len(class_names) + 15))

for i, true_name in enumerate(class_names):
    print(f"{true_name:>6} |", end="")
    for j in range(len(class_names)):
        count = np.sum((all_labels == i) & (all_predictions == j))
        if i == j:
            print(f"  [{count:>3}]", end="")  # Highlight diagonal
        elif count > 0:
            print(f"   {count:>3}", end="")    # Show misclassifications
        else:
            print(f"     .", end="")            # Clean zeros
    print(f"  | True: {true_name}")

# Per-class accuracy
print(f"\nPer-class accuracy:")
for i, name in enumerate(class_names):
    mask = all_labels == i
    if mask.sum() > 0:
        acc = np.sum(all_predictions[mask] == i) / mask.sum()
        total = mask.sum()
        correct = np.sum(all_predictions[mask] == i)
        print(f"  {name}: {acc:.1%} ({correct}/{total})")

overall_acc = np.sum(all_predictions == all_labels) / len(all_labels)
print(f"\nOverall accuracy: {overall_acc:.1%}")

# Show worst confusions
print(f"\nTop confusions (most common mistakes):")
confusions = []
for i in range(len(class_names)):
    for j in range(len(class_names)):
        if i != j:
            count = np.sum((all_labels == i) & (all_predictions == j))
            if count > 0:
                confusions.append((class_names[i], class_names[j], count))
confusions.sort(key=lambda x: x[2], reverse=True)
for true_cls, pred_cls, count in confusions[:10]:
    print(f"  {true_cls} misclassified as {pred_cls}: {count} times")

# ── Step 11: Save ────────────────────────────────────────────────────────────
model.save(MODEL_NAME)
print(f"\nSaved {MODEL_NAME}")
print(f"Saved {CLASSES_NAME}")
print("Download both files from the Colab files panel on the left.")

ModuleNotFoundError: No module named 'tensorflow'

# Pip


In [ ]:
!pip install tensorflow

# April 12 27 classes training for combined model

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ASL 27-Class Model Training — Production Build
# ══════════════════════════════════════════════════════════════════════════════
# Weights:    asl_model4_27class.weights.h5
# Classes:    class_names4.json
# Resolution: 160x160
# Classes:    A-Z + space (27 classes, all letters included)
# Data:       All Kaggle images (~3000/class) + webcam images (appended in-place)
# Use with:   live_asl5.py
# ══════════════════════════════════════════════════════════════════════════════

# ── Step 0: Cleanup ───────────────────────────────────────────────────────────
import os
import shutil

for folder in ['asl_alphabet_train', 'asl_alphabet_test', 'webcam_training_data_224']:
    if os.path.exists(folder):
        shutil.rmtree(folder)
        print(f"Cleaned up old {folder}")

# ── Step 1: Unzip datasets ───────────────────────────────────────────────────
!unzip -q 'archive (1).zip'
!unzip -q 'webcam_training_data_224.zip'
!echo "Unzip done."

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import json
import os
import shutil
import gc
import numpy as np

# ── Config ────────────────────────────────────────────────────────────────────
IMAGE_SIZE       = (160, 160)
BATCH_SIZE       = 32
EPOCHS_FROZEN    = 50
EPOCHS_FINETUNE  = 25
KAGGLE_DIR       = 'asl_alphabet_train/asl_alphabet_train'
WEBCAM_DIR       = 'webcam_training_data_224'
WEIGHTS_NAME     = 'asl_model4_27class.weights.h5'
CLASSES_NAME     = 'class_names4.json'
# ─────────────────────────────────────────────────────────────────────────────

if not os.path.exists(KAGGLE_DIR):
    print("Expected directory not found. Available directories:")
    os.system('find . -maxdepth 4 -type d')
    raise Exception("Update KAGGLE_DIR to match the correct path shown above")

print(f"Kaggle dataset found at: {KAGGLE_DIR}")

# ── Remove unwanted classes (only nothing and del) ────────────────────────────
for remove_class in ['nothing', 'del']:
    class_path = os.path.join(KAGGLE_DIR, remove_class)
    if os.path.exists(class_path):
        shutil.rmtree(class_path)
        print(f"Removed '{remove_class}' class")

# ── Merge webcam images directly INTO Kaggle folders (no duplication) ─────────
webcam_total = 0
if os.path.exists(WEBCAM_DIR):
    print(f"\nMerging webcam images into Kaggle dataset...")
    for cls in sorted(os.listdir(WEBCAM_DIR)):
        webcam_cls = os.path.join(WEBCAM_DIR, cls)
        kaggle_cls = os.path.join(KAGGLE_DIR, cls)
        if not os.path.isdir(webcam_cls):
            continue
        if not os.path.exists(kaggle_cls):
            os.makedirs(kaggle_cls, exist_ok=True)
        images = [f for f in os.listdir(webcam_cls) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        for img in images:
            shutil.copy(os.path.join(webcam_cls, img), os.path.join(kaggle_cls, f"webcam_{img}"))
        webcam_total += len(images)
        print(f"  {cls}: added {len(images)} webcam images")
    print(f"Total webcam images merged: {webcam_total}")
else:
    print(f"\nWARNING: '{WEBCAM_DIR}' not found. Training with Kaggle data only.")

# ── Verify final class counts ─────────────────────────────────────────────────
print(f"\nFinal dataset:")
total_images = 0
for cls in sorted(os.listdir(KAGGLE_DIR)):
    cls_path = os.path.join(KAGGLE_DIR, cls)
    if os.path.isdir(cls_path):
        count = len([f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        print(f"  {cls}: {count} images")
        total_images += count
print(f"Total: {total_images} images")

gc.collect()

# ── Step 2: Load datasets ────────────────────────────────────────────────────
print(f"\nLoading dataset at {IMAGE_SIZE} resolution...")
full_dataset = tf.keras.utils.image_dataset_from_directory(
    KAGGLE_DIR,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    validation_split=0.2,
    subset='both',
    seed=42
)

train_dataset, validation_dataset = full_dataset

class_names = train_dataset.class_names
NUM_CLASSES = len(class_names)
print(f"\nClasses ({NUM_CLASSES}): {class_names}")

with open(CLASSES_NAME, 'w') as f:
    json.dump(class_names, f)
print(f"Saved {CLASSES_NAME}")

# ── Step 3: Optimized pipeline ────────────────────────────────────────────────
# .cache() keeps decoded images in RAM after first epoch — big speed boost.
# Colab Pro should handle this. If it crashes, remove .cache()
train_dataset = (
    train_dataset
    .cache()
    .shuffle(1000)
    .prefetch(tf.data.AUTOTUNE)
)

validation_dataset = (
    validation_dataset
    .cache()
    .prefetch(tf.data.AUTOTUNE)
)

# ── Step 4: Augmentation ─────────────────────────────────────────────────────
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.10),
    layers.RandomZoom(0.15),
    layers.RandomBrightness(0.3),
    layers.RandomContrast(0.3),
    layers.RandomTranslation(0.1, 0.1),
], name="augmentation")

# ── Step 5: Build model ──────────────────────────────────────────────────────
base_model = keras.applications.MobileNetV2(
    input_shape=IMAGE_SIZE + (3,),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

inputs = keras.Input(shape=IMAGE_SIZE + (3,))
x = data_augmentation(inputs)
x = keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(512, activation='relu')(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = keras.Model(inputs, outputs)

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ── Step 6: Phase 1 — Frozen backbone ────────────────────────────────────────
print("\n" + "=" * 60)
print("PHASE 1: Training with frozen backbone")
print("=" * 60)

history = model.fit(
    train_dataset,
    epochs=EPOCHS_FROZEN,
    validation_data=validation_dataset,
    callbacks=[
        keras.callbacks.EarlyStopping(
            patience=7,
            restore_best_weights=True,
            monitor='val_accuracy',
            verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            factor=0.5,
            patience=3,
            min_lr=1e-6,
            verbose=1
        ),
    ],
    verbose=1
)

results = model.evaluate(validation_dataset)
print(f"\nPhase 1 — Val Loss: {results[0]:.4f} | Val Accuracy: {results[1]:.4f}")

# Save Phase 1 weights as backup
model.save_weights('asl_model4_phase1_backup.weights.h5')
print("Phase 1 weights saved as backup.")

# ── Step 7: Phase 2 — Fine-tune backbone ─────────────────────────────────────
print("\n" + "=" * 60)
print("PHASE 2: Fine-tuning top 30 backbone layers")
print("=" * 60)

base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

trainable_count = sum(1 for l in base_model.layers if l.trainable)
frozen_count = sum(1 for l in base_model.layers if not l.trainable)
print(f"Backbone: {trainable_count} trainable, {frozen_count} frozen")

model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_finetune = model.fit(
    train_dataset,
    epochs=EPOCHS_FINETUNE,
    validation_data=validation_dataset,
    callbacks=[
        keras.callbacks.EarlyStopping(
            patience=5,
            restore_best_weights=True,
            monitor='val_accuracy',
            verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            factor=0.5,
            patience=2,
            min_lr=1e-7,
            verbose=1
        ),
    ],
    verbose=1
)

# ── Step 8: Final evaluation ─────────────────────────────────────────────────
print("\n" + "=" * 60)
print("FINAL EVALUATION")
print("=" * 60)

results = model.evaluate(validation_dataset)
print(f"\nFinal Val Loss: {results[0]:.4f} | Final Val Accuracy: {results[1]:.4f}")

# ── Step 9: Confusion matrix ─────────────────────────────────────────────────
print("\n" + "=" * 60)
print("CONFUSION MATRIX")
print("=" * 60)

all_predictions = []
all_labels = []

for images, labels in validation_dataset:
    preds = model.predict(images, verbose=0)
    all_predictions.extend(np.argmax(preds, axis=1))
    all_labels.extend(np.argmax(labels.numpy(), axis=1))

all_predictions = np.array(all_predictions)
all_labels = np.array(all_labels)

# Print confusion matrix
print(f"\n{'':>8}", end="")
for name in class_names:
    print(f"{name:>5}", end="")
print("  <- Predicted")
print("-" * (8 + 5 * len(class_names) + 15))

for i, true_name in enumerate(class_names):
    print(f"{true_name:>6} |", end="")
    for j in range(len(class_names)):
        count = np.sum((all_labels == i) & (all_predictions == j))
        if i == j:
            print(f" [{count:>2}]", end="")
        elif count > 0:
            print(f"  {count:>2}", end="")
        else:
            print(f"    .", end="")
    print(f"  | {true_name}")

# Per-class accuracy sorted worst to best
print(f"\nPer-class accuracy (sorted worst to best):")
class_accs = []
for i, name in enumerate(class_names):
    mask = all_labels == i
    if mask.sum() > 0:
        acc = np.sum(all_predictions[mask] == i) / mask.sum()
        total = int(mask.sum())
        correct = int(np.sum(all_predictions[mask] == i))
        class_accs.append((name, acc, correct, total))

class_accs.sort(key=lambda x: x[1])
for name, acc, correct, total in class_accs:
    bar = "#" * int(acc * 20) + "." * (20 - int(acc * 20))
    print(f"  {name}: [{bar}] {acc:.1%} ({correct}/{total})")

overall_acc = np.sum(all_predictions == all_labels) / len(all_labels)
print(f"\nOverall accuracy: {overall_acc:.1%}")

# Worst confusions
print(f"\nTop 20 confusions (most common mistakes):")
confusions = []
for i in range(len(class_names)):
    for j in range(len(class_names)):
        if i != j:
            count = int(np.sum((all_labels == i) & (all_predictions == j)))
            if count > 0:
                confusions.append((class_names[i], class_names[j], count))
confusions.sort(key=lambda x: x[2], reverse=True)
for true_cls, pred_cls, count in confusions[:20]:
    print(f"  {true_cls} predicted as {pred_cls}: {count} times")

# Identify weak letters
print(f"\nLetters below 90% accuracy (capture more webcam data for these):")
weak_letters = [name for name, acc, _, _ in class_accs if acc < 0.90]
if weak_letters:
    print(f"  {', '.join(weak_letters)}")
else:
    print(f"  None — all classes above 90%!")

# ── Step 10: Save weights ────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("SAVING")
print("=" * 60)

model.save_weights(WEIGHTS_NAME)
print(f"Saved {WEIGHTS_NAME}")
print(f"Saved {CLASSES_NAME}")
print(f"\nDownload from the Colab files panel:")
print(f"  1. {WEIGHTS_NAME}")
print(f"  2. {CLASSES_NAME}")
print(f"\nUse with live_asl5.py")

Cleaned up old asl_alphabet_train
Cleaned up old asl_alphabet_test
Cleaned up old webcam_training_data_224
replace __MACOSX/._webcam_training_data_224? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
Unzip done.


/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


Kaggle dataset found at: asl_alphabet_train/asl_alphabet_train
Removed 'nothing' class
Removed 'del' class

Merging webcam images into Kaggle dataset...
  A: added 300 webcam images
  B: added 300 webcam images
  C: added 300 webcam images
  D: added 300 webcam images
  E: added 300 webcam images
  F: added 300 webcam images
  G: added 300 webcam images
  H: added 300 webcam images
  I: added 300 webcam images
  J: added 300 webcam images
  K: added 300 webcam images
  L: added 300 webcam images
  M: added 300 webcam images
  N: added 300 webcam images
  O: added 300 webcam images
  P: added 300 webcam images
  Q: added 300 webcam images
  R: added 300 webcam images
  S: added 300 webcam images
  T: added 300 webcam images
  U: added 300 webcam images
  V: added 300 webcam images
  W: added 300 webcam images
  X: added 300 webcam images
  Y: added 300 webcam images
  Z: added 300 webcam images
  space: added 300 webcam images
Total webcam images merged: 8100

Final dataset:
  A: 3300 i

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ augmentation (Sequential)       │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_160            │ (None, 5, 5, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       655,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 27)             │         6,939 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,052,123 (11.64 MB)

 Trainable params: 794,139 (3.03 MB)

 Non-trainable params: 2,257,984 (8.61 MB)


PHASE 1: Training with frozen backbone
Epoch 1/50
2228/2228 ━━━━━━━━━━━━━━━━━━━━ 172s 76ms/step - accuracy: 0.6136 - loss: 1.2122 - val_accuracy: 0.8561 - val_loss: 0.4343 - learning_rate: 0.0010
Epoch 2/50
2228/2228 ━━━━━━━━━━━━━━━━━━━━ 166s 75ms/step - accuracy: 0.7534 - loss: 0.7606 - val_accuracy: 0.8995 - val_loss: 0.3074 - learning_rate: 0.0010
Epoch 3/50
2228/2228 ━━━━━━━━━━━━━━━━━━━━ 166s 75ms/step - accuracy: 0.7821 - loss: 0.6709 - val_accuracy: 0.9121 - val_loss: 0.2801 - learning_rate: 0.0010
Epoch 4/50
2228/2228 ━━━━━━━━━━━━━━━━━━━━ 166s 75ms/step - accuracy: 0.8007 - loss: 0.6213 - val_accuracy: 0.9246 - val_loss: 0.2371 - learning_rate: 0.0010
Epoch 5/50
2228/2228 ━━━━━━━━━━━━━━━━━━━━ 166s 75ms/step - accuracy: 0.8133 - loss: 0.5835 - val_accuracy: 0.9222 - val_loss: 0.2508 - learning_rate: 0.0010
Epoch 6/50
2228/2228 ━━━━━━━━━━━━━━━━━━━━ 166s 75ms/step - accuracy: 0.8197 - loss: 0.5600 - val_accuracy: 0.9122 - val_loss: 0.2605 - learning_rate: 0.0010
Epoch 7/50
2228/22

# April13th, 1000 images training 224

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ASL 27-Class Model Training — 224x224 (Memory Optimized)
# ══════════════════════════════════════════════════════════════════════════════
# Weights:    asl_model4_27class.weights.h5
# Classes:    class_names4.json
# Resolution: 224x224 (MobileNetV2 native)
# Classes:    A-Z + space (27 classes, all letters included)
# Data:       Full Kaggle (~3000/class) + both webcam sets merged in
# Use with:   live_asl5.py
# ══════════════════════════════════════════════════════════════════════════════

# ── Step 0: Cleanup ───────────────────────────────────────────────────────────
import os
import shutil

for folder in ['asl_alphabet_train', 'asl_alphabet_test',
               'webcam_training_data_224', 'webcam_training_data_224_v2']:
    if os.path.exists(folder):
        shutil.rmtree(folder)
        print(f"Cleaned up old {folder}")

# ── Step 1: Unzip all datasets ───────────────────────────────────────────────
!unzip -q 'archive (1).zip'
!unzip -q 'webcam_training_data_224.zip'
!unzip -q 'webcam_training_data_224_v21000images.zip'
!echo "Unzip done."

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import json
import os
import shutil
import gc
import numpy as np

# ── Enable mixed precision (float16 — cuts memory nearly in half) ─────────────
keras.mixed_precision.set_global_policy('mixed_float16')
print("Mixed precision enabled: training in float16")

# ── Config ────────────────────────────────────────────────────────────────────
IMAGE_SIZE       = (224, 224)
BATCH_SIZE       = 16
EPOCHS_FROZEN    = 50
EPOCHS_FINETUNE  = 25
KAGGLE_DIR       = 'asl_alphabet_train/asl_alphabet_train'
WEBCAM_DIRS      = ['webcam_training_data_224', 'webcam_training_data_224_v21000images']
WEIGHTS_NAME     = 'asl_model4_27class.weights.h5'
CLASSES_NAME     = 'class_names4.json'
# ─────────────────────────────────────────────────────────────────────────────

if not os.path.exists(KAGGLE_DIR):
    print("Expected directory not found. Available directories:")
    os.system('find . -maxdepth 4 -type d')
    raise Exception("Update KAGGLE_DIR to match the correct path shown above")

print(f"Kaggle dataset found at: {KAGGLE_DIR}")

# ── Remove unwanted classes ───────────────────────────────────────────────────
for remove_class in ['nothing', 'del']:
    class_path = os.path.join(KAGGLE_DIR, remove_class)
    if os.path.exists(class_path):
        shutil.rmtree(class_path)
        print(f"Removed '{remove_class}' class")

# ── Merge ALL webcam folders into Kaggle (no duplication) ─────────────────────
webcam_grand_total = 0
for webcam_dir in WEBCAM_DIRS:
    if not os.path.exists(webcam_dir):
        print(f"\n'{webcam_dir}' not found — skipping.")
        continue
    print(f"\nMerging {webcam_dir} into Kaggle dataset...")
    dir_total = 0
    for cls in sorted(os.listdir(webcam_dir)):
        webcam_cls = os.path.join(webcam_dir, cls)
        kaggle_cls = os.path.join(KAGGLE_DIR, cls)
        if not os.path.isdir(webcam_cls):
            continue
        if not os.path.exists(kaggle_cls):
            os.makedirs(kaggle_cls, exist_ok=True)
        images = [f for f in os.listdir(webcam_cls) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        # Prefix with folder name to avoid filename collisions between sets
        prefix = webcam_dir.replace('/', '_')
        for img in images:
            shutil.copy(os.path.join(webcam_cls, img),
                        os.path.join(kaggle_cls, f"{prefix}_{img}"))
        dir_total += len(images)
        print(f"  {cls}: added {len(images)} images")
    print(f"  Subtotal from {webcam_dir}: {dir_total}")
    webcam_grand_total += dir_total

print(f"\nTotal webcam images merged: {webcam_grand_total}")

# ── Verify final class counts ─────────────────────────────────────────────────
print(f"\nFinal dataset:")
total_images = 0
for cls in sorted(os.listdir(KAGGLE_DIR)):
    cls_path = os.path.join(KAGGLE_DIR, cls)
    if os.path.isdir(cls_path):
        count = len([f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        print(f"  {cls}: {count} images")
        total_images += count
print(f"Total: {total_images} images")

gc.collect()

# ── Step 2: Load datasets ────────────────────────────────────────────────────
print(f"\nLoading dataset at {IMAGE_SIZE} resolution...")
full_dataset = tf.keras.utils.image_dataset_from_directory(
    KAGGLE_DIR,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    validation_split=0.2,
    subset='both',
    seed=42
)

train_dataset, validation_dataset = full_dataset

class_names = train_dataset.class_names
NUM_CLASSES = len(class_names)
print(f"\nClasses ({NUM_CLASSES}): {class_names}")

with open(CLASSES_NAME, 'w') as f:
    json.dump(class_names, f)
print(f"Saved {CLASSES_NAME}")

# ── Step 3: Lightweight pipeline (NO .cache() — saves RAM) ───────────────────
train_dataset = (
    train_dataset
    .shuffle(256)
    .prefetch(tf.data.AUTOTUNE)
)

validation_dataset = (
    validation_dataset
    .prefetch(tf.data.AUTOTUNE)
)

# ── Step 4: Augmentation ─────────────────────────────────────────────────────
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.10),
    layers.RandomZoom(0.15),
    layers.RandomBrightness(0.3),
    layers.RandomContrast(0.3),
    layers.RandomTranslation(0.1, 0.1),
], name="augmentation")

# ── Step 5: Build model ──────────────────────────────────────────────────────
base_model = keras.applications.MobileNetV2(
    input_shape=IMAGE_SIZE + (3,),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

inputs = keras.Input(shape=IMAGE_SIZE + (3,))
x = data_augmentation(inputs)
x = keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(512, activation='relu')(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.3)(x)
# float32 output for numerical stability with mixed precision
outputs = layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)

model = keras.Model(inputs, outputs)

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ── Step 6: Phase 1 — Frozen backbone ────────────────────────────────────────
print("\n" + "=" * 60)
print("PHASE 1: Training with frozen backbone")
print("=" * 60)

history = model.fit(
    train_dataset,
    epochs=EPOCHS_FROZEN,
    validation_data=validation_dataset,
    callbacks=[
        keras.callbacks.EarlyStopping(
            patience=7,
            restore_best_weights=True,
            monitor='val_accuracy',
            verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            factor=0.5,
            patience=3,
            min_lr=1e-6,
            verbose=1
        ),
    ],
    verbose=1
)

results = model.evaluate(validation_dataset)
print(f"\nPhase 1 — Val Loss: {results[0]:.4f} | Val Accuracy: {results[1]:.4f}")

# Save Phase 1 weights as backup
model.save_weights('asl_model4_phase1_backup.weights.h5')
print("Phase 1 weights saved as backup.")

# ── Step 7: Phase 2 — Fine-tune top 20 backbone layers ───────────────────────
print("\n" + "=" * 60)
print("PHASE 2: Fine-tuning top 20 backbone layers")
print("=" * 60)

base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

trainable_count = sum(1 for l in base_model.layers if l.trainable)
frozen_count = sum(1 for l in base_model.layers if not l.trainable)
print(f"Backbone: {trainable_count} trainable, {frozen_count} frozen")

model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_finetune = model.fit(
    train_dataset,
    epochs=EPOCHS_FINETUNE,
    validation_data=validation_dataset,
    callbacks=[
        keras.callbacks.EarlyStopping(
            patience=5,
            restore_best_weights=True,
            monitor='val_accuracy',
            verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            factor=0.5,
            patience=2,
            min_lr=1e-7,
            verbose=1
        ),
    ],
    verbose=1
)

# ── Step 8: Final evaluation ─────────────────────────────────────────────────
print("\n" + "=" * 60)
print("FINAL EVALUATION")
print("=" * 60)

results = model.evaluate(validation_dataset)
print(f"\nFinal Val Loss: {results[0]:.4f} | Final Val Accuracy: {results[1]:.4f}")

# ── Step 9: Confusion matrix ─────────────────────────────────────────────────
print("\n" + "=" * 60)
print("CONFUSION MATRIX")
print("=" * 60)

all_predictions = []
all_labels = []

for images, labels in validation_dataset:
    preds = model.predict(images, verbose=0)
    all_predictions.extend(np.argmax(preds, axis=1))
    all_labels.extend(np.argmax(labels.numpy(), axis=1))

all_predictions = np.array(all_predictions)
all_labels = np.array(all_labels)

print(f"\n{'':>8}", end="")
for name in class_names:
    print(f"{name:>5}", end="")
print("  <- Predicted")
print("-" * (8 + 5 * len(class_names) + 15))

for i, true_name in enumerate(class_names):
    print(f"{true_name:>6} |", end="")
    for j in range(len(class_names)):
        count = np.sum((all_labels == i) & (all_predictions == j))
        if i == j:
            print(f" [{count:>2}]", end="")
        elif count > 0:
            print(f"  {count:>2}", end="")
        else:
            print(f"    .", end="")
    print(f"  | {true_name}")

print(f"\nPer-class accuracy (sorted worst to best):")
class_accs = []
for i, name in enumerate(class_names):
    mask = all_labels == i
    if mask.sum() > 0:
        acc = np.sum(all_predictions[mask] == i) / mask.sum()
        total = int(mask.sum())
        correct = int(np.sum(all_predictions[mask] == i))
        class_accs.append((name, acc, correct, total))

class_accs.sort(key=lambda x: x[1])
for name, acc, correct, total in class_accs:
    bar = "#" * int(acc * 20) + "." * (20 - int(acc * 20))
    print(f"  {name}: [{bar}] {acc:.1%} ({correct}/{total})")

overall_acc = np.sum(all_predictions == all_labels) / len(all_labels)
print(f"\nOverall accuracy: {overall_acc:.1%}")

print(f"\nTop 20 confusions (most common mistakes):")
confusions = []
for i in range(len(class_names)):
    for j in range(len(class_names)):
        if i != j:
            count = int(np.sum((all_labels == i) & (all_predictions == j)))
            if count > 0:
                confusions.append((class_names[i], class_names[j], count))
confusions.sort(key=lambda x: x[2], reverse=True)
for true_cls, pred_cls, count in confusions[:20]:
    print(f"  {true_cls} predicted as {pred_cls}: {count} times")

print(f"\nLetters below 90% accuracy (capture more webcam data for these):")
weak_letters = [name for name, acc, _, _ in class_accs if acc < 0.90]
if weak_letters:
    print(f"  {', '.join(weak_letters)}")
else:
    print(f"  None — all classes above 90%!")

# ── Step 10: Save weights ────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("SAVING")
print("=" * 60)

model.save_weights(WEIGHTS_NAME)
print(f"Saved {WEIGHTS_NAME}")
print(f"Saved {CLASSES_NAME}")
print(f"\nDownload from the Colab files panel:")
print(f"  1. {WEIGHTS_NAME}")
print(f"  2. {CLASSES_NAME}")
print(f"\nUse with live_asl5.py")

Cleaned up old asl_alphabet_train
Cleaned up old asl_alphabet_test
Cleaned up old webcam_training_data_224
replace __MACOSX/._webcam_training_data_224? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
replace __MACOSX/._webcam_training_data_224_v21000images? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
Unzip done.
Mixed precision enabled: training in float16
Kaggle dataset found at: asl_alphabet_train/asl_alphabet_train
Removed 'nothing' class
Removed 'del' class

Merging webcam_training_data_224 into Kaggle dataset...
  A: added 300 images
  B: added 300 images
  C: added 300 images
  D: added 300 images
  E: added 300 images
  F: added 300 images
  G: added 300 images
  H: added 300 images
  I: added 300 images
  J: added 300 images
  K: added 300 images
  L: added 300 images
  M: added 300 images
  N: added 300 images
  O: added 300 images
  P: added 300 images
  Q: added 300 images
  R: added 300 images
  S: added 300 images
  T: added 300 images
  U: added 300 images
  V: added 300 images
  W: 

Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_10 (InputLayer)     │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ augmentation (Sequential)       │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide_3 (TrueDivide)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract_3 (Subtract)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 512)            │       655,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 27)             │         6,939 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,052,123 (11.64 MB)

 Trainable params: 794,139 (3.03 MB)

 Non-trainable params: 2,257,984 (8.61 MB)


PHASE 1: Training with frozen backbone
Epoch 1/50
6280/6280 ━━━━━━━━━━━━━━━━━━━━ 800s 127ms/step - accuracy: 0.6487 - loss: 1.1044 - val_accuracy: 0.8683 - val_loss: 0.3870 - learning_rate: 0.0010
Epoch 2/50
6280/6280 ━━━━━━━━━━━━━━━━━━━━ 800s 127ms/step - accuracy: 0.7635 - loss: 0.7370 - val_accuracy: 0.8966 - val_loss: 0.3064 - learning_rate: 0.0010
Epoch 3/50
6280/6280 ━━━━━━━━━━━━━━━━━━━━ 801s 127ms/step - accuracy: 0.7896 - loss: 0.6568 - val_accuracy: 0.9239 - val_loss: 0.2429 - learning_rate: 0.0010
Epoch 4/50
6280/6280 ━━━━━━━━━━━━━━━━━━━━ 801s 127ms/step - accuracy: 0.8069 - loss: 0.6167 - val_accuracy: 0.9138 - val_loss: 0.2689 - learning_rate: 0.0010
Epoch 5/50
6280/6280 ━━━━━━━━━━━━━━━━━━━━ 800s 127ms/step - accuracy: 0.8137 - loss: 0.5978 - val_accuracy: 0.9125 - val_loss: 0.2601 - learning_rate: 0.0010
Epoch 6/50
6280/6280 ━━━━━━━━━━━━━━━━━━━━ 801s 127ms/step - accuracy: 0.8204 - loss: 0.5835 - val_accuracy: 0.9275 - val_loss: 0.2262 - learning_rate: 0.0010
Epoch 7/50
6

# Final model runApril26

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ASL 27-Class Model Training — Final (256×256, Colab Pro+ A100, split-phase)
# ══════════════════════════════════════════════════════════════════════════════
#
# Changes from train_asl_27class_224.py:
#   * 256×256 resolution (divisible by MobileNetV2's total stride of 32).
#   * SPLIT-PHASE TRAINING:
#       - Phase 1 (frozen backbone): Kaggle + webcam → trains the classifier
#         head on a large diverse pool. Volume matters here, distribution
#         doesn't, so Kaggle's bulk helps.
#       - Phase 2 (fine-tune top 40 layers): webcam ONLY → adapts the
#         backbone features specifically to your deployment distribution.
#         Including Kaggle in this phase pulls features back toward Kaggle-
#         land and works against live-webcam accuracy.
#     This is a strict upgrade over training both phases on the merged pool.
#   * AdamW optimizer with weight_decay=1e-4.
#   * CosineDecay learning-rate schedule (replaces ReduceLROnPlateau).
#   * CategoricalCrossentropy with label_smoothing=0.1.
#   * Removed RandomFlip("horizontal") — ASL signs are hand-specific and
#     horizontal flipping corrupts ~half the training data. THIS IS A BUG FIX,
#     not a tuning change. Do not re-introduce.
#   * Shuffle buffer increased and applied BEFORE batching.
#   * Fine-tunes top 40 layers in Phase 2 (was 20).
#   * BatchNorm in backbone stays in inference mode during fine-tuning
#     (`training=False`) — intentional. Do not unfreeze BN at batch_size ≤ 32.
#   * Optional SESSION-SEPARATED webcam validation set (WEBCAM_VAL_DIR).
#   * Deterministic seeding for reproducibility.
#
# To use on Colab Pro+:
#   - Upload archive (1).zip (Kaggle), webcam training zip(s), and ideally a
#     separate webcam validation zip from a different session/location.
#   - Runtime → Change runtime type → A100 GPU, High-RAM.
#   - Paste this whole file as a single cell.
#
# Expect validation accuracy to be LOWER than your old 99.7% — probably
# 88–95% on a session-separated webcam val set. That's not a regression; it's
# finally measuring what matters.

import os
import shutil
import random

# Clean up any leftover unzipped folders from prior runs
for folder in ['asl_alphabet_train', 'asl_alphabet_test',
               'webcam_training_data_224',
               'webcam_training_data_224_v2',
               'webcam_training_data_224_v21000images',
               'webcam_training_data_Final',
               'webcam_val_Final',
               'webcam_only_pool']:
    if os.path.exists(folder):
        shutil.rmtree(folder)
        print(f"Cleaned up old {folder}")

# Adjust the unzip commands to match your actual file names on Colab
!unzip -q 'archive (1).zip'
!unzip -q 'webcam_training_data_Final.zip'
!unzip -q 'webcam_val_Final.zip'
!echo "Unzip done."

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import json
import gc
import numpy as np

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
keras.utils.set_random_seed(SEED)

# ── Mixed precision (still useful for memory headroom even on A100 at 256) ──
keras.mixed_precision.set_global_policy('mixed_float16')
print("Mixed precision enabled: training in float16")

# ══════════════════════════════════════════════════════════════════════════════
# Config
# ══════════════════════════════════════════════════════════════════════════════

IMAGE_SIZE          = (256, 256)
BATCH_SIZE          = 32
EPOCHS_FROZEN       = 30
EPOCHS_FINETUNE     = 20
FINETUNE_LAYERS     = 40
LABEL_SMOOTHING     = 0.1
WEIGHT_DECAY        = 1e-4
INITIAL_LR_FROZEN   = 1e-3
INITIAL_LR_FINETUNE = 1e-5
DROPOUT_RATE        = 0.3

KAGGLE_DIR       = 'asl_alphabet_train/asl_alphabet_train'
WEBCAM_DIRS      = ['webcam_training_data_Final']
WEBCAM_VAL_DIR   = 'webcam_val_Final'   # session-separated held-out validation
WEBCAM_ONLY_DIR  = 'webcam_only_pool'  # auto-built; used for Phase 2

WEIGHTS_NAME    = 'asl_model_Final.weights.h5'
CLASSES_NAME    = 'class_names_Final.json'
PHASE1_BACKUP   = 'asl_model_Final_phase1_backup.weights.h5'

EXPECTED_CLASSES = sorted([chr(65 + i) for i in range(26)] + ['space'])

# ══════════════════════════════════════════════════════════════════════════════
# Sanity checks on dataset paths
# ══════════════════════════════════════════════════════════════════════════════

if not os.path.exists(KAGGLE_DIR):
    print("Expected directory not found. Available directories:")
    os.system('find . -maxdepth 4 -type d')
    raise Exception(f"Update KAGGLE_DIR to match the correct path shown above (was {KAGGLE_DIR})")

print(f"Kaggle dataset found at: {KAGGLE_DIR}")

for remove_class in ['nothing', 'del']:
    class_path = os.path.join(KAGGLE_DIR, remove_class)
    if os.path.exists(class_path):
        shutil.rmtree(class_path)
        print(f"Removed '{remove_class}' class")

# ══════════════════════════════════════════════════════════════════════════════
# Build webcam-only pool (Phase 2 source) BEFORE merging into Kaggle
# ══════════════════════════════════════════════════════════════════════════════

print("\nBuilding webcam-only pool for Phase 2 fine-tuning...")
os.makedirs(WEBCAM_ONLY_DIR, exist_ok=True)

webcam_only_total = 0
for webcam_dir in WEBCAM_DIRS:
    if not os.path.exists(webcam_dir):
        print(f"  '{webcam_dir}' not found — skipping.")
        continue
    for cls in sorted(os.listdir(webcam_dir)):
        webcam_cls = os.path.join(webcam_dir, cls)
        if not os.path.isdir(webcam_cls):
            continue
        target_cls = os.path.join(WEBCAM_ONLY_DIR, cls)
        os.makedirs(target_cls, exist_ok=True)
        images = [f for f in os.listdir(webcam_cls)
                  if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        prefix = webcam_dir.replace('/', '_')
        for img in images:
            shutil.copy(os.path.join(webcam_cls, img),
                        os.path.join(target_cls, f"{prefix}_{img}"))
        webcam_only_total += len(images)

if webcam_only_total == 0:
    raise Exception(
        "No webcam images found. Phase 2 requires webcam data. "
        "Check that WEBCAM_DIRS points to valid folders and that the "
        "uploaded zip(s) extracted properly."
    )

# Ensure all 27 expected classes have folders so class indices align with
# Phase 1's dataset. Empty folders are fine — they just contribute 0 samples.
missing_classes = []
for cls in EXPECTED_CLASSES:
    cls_path = os.path.join(WEBCAM_ONLY_DIR, cls)
    if not os.path.exists(cls_path) or len(os.listdir(cls_path)) == 0:
        os.makedirs(cls_path, exist_ok=True)
        missing_classes.append(cls)

if missing_classes:
    print(f"  WARNING: webcam pool has no samples for: {missing_classes}")
    print("  These classes will not be fine-tuned in Phase 2. Capture them "
          "before relying on the model for those letters.")

print(f"Webcam-only pool: {webcam_only_total} images across "
      f"{len(EXPECTED_CLASSES)} class folders")

# ══════════════════════════════════════════════════════════════════════════════
# Merge webcam into Kaggle for Phase 1 (existing behavior)
# ══════════════════════════════════════════════════════════════════════════════

webcam_grand_total = 0
for webcam_dir in WEBCAM_DIRS:
    if not os.path.exists(webcam_dir):
        continue
    print(f"\nMerging {webcam_dir} into Kaggle dataset (for Phase 1)...")
    dir_total = 0
    for cls in sorted(os.listdir(webcam_dir)):
        webcam_cls = os.path.join(webcam_dir, cls)
        kaggle_cls = os.path.join(KAGGLE_DIR, cls)
        if not os.path.isdir(webcam_cls):
            continue
        if not os.path.exists(kaggle_cls):
            os.makedirs(kaggle_cls, exist_ok=True)
        images = [f for f in os.listdir(webcam_cls)
                  if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        prefix = webcam_dir.replace('/', '_')
        for img in images:
            shutil.copy(os.path.join(webcam_cls, img),
                        os.path.join(kaggle_cls, f"{prefix}_{img}"))
        dir_total += len(images)
        print(f"  {cls}: added {len(images)} images")
    print(f"  Subtotal from {webcam_dir}: {dir_total}")
    webcam_grand_total += dir_total
print(f"\nTotal webcam images merged into Phase 1 pool: {webcam_grand_total}")

# Final summary
print("\nFinal Phase 1 training pool (Kaggle + webcam):")
total_images = 0
for cls in sorted(os.listdir(KAGGLE_DIR)):
    cls_path = os.path.join(KAGGLE_DIR, cls)
    if os.path.isdir(cls_path):
        count = len([f for f in os.listdir(cls_path)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        print(f"  {cls}: {count} images")
        total_images += count
print(f"Phase 1 total: {total_images} images")
print(f"Phase 2 total: {webcam_only_total} images (webcam only)")

gc.collect()

# ══════════════════════════════════════════════════════════════════════════════
# Load datasets
# ══════════════════════════════════════════════════════════════════════════════

print(f"\nLoading Phase 1 (Kaggle + webcam) training set at {IMAGE_SIZE}...")

if WEBCAM_VAL_DIR is not None and os.path.exists(WEBCAM_VAL_DIR):
    print(f"Using SESSION-SEPARATED validation from {WEBCAM_VAL_DIR}")
    print("This is the realistic measurement of deployment accuracy.")

    train_dataset_p1 = tf.keras.utils.image_dataset_from_directory(
        KAGGLE_DIR,
        image_size=IMAGE_SIZE,
        batch_size=None,
        label_mode='categorical',
        shuffle=True,
        seed=SEED,
    )
    validation_dataset = tf.keras.utils.image_dataset_from_directory(
        WEBCAM_VAL_DIR,
        image_size=IMAGE_SIZE,
        batch_size=None,
        label_mode='categorical',
        shuffle=False,
    )
else:
    print("WARNING: WEBCAM_VAL_DIR not set or not found.")
    print("Falling back to random 80/20 split — val accuracy will be inflated")
    print("because adjacent webcam frames leak across train/val.")

    full_dataset = tf.keras.utils.image_dataset_from_directory(
        KAGGLE_DIR,
        image_size=IMAGE_SIZE,
        batch_size=None,
        label_mode='categorical',
        validation_split=0.2,
        subset='both',
        seed=SEED,
    )
    train_dataset_p1, validation_dataset = full_dataset

class_names = train_dataset_p1.class_names
NUM_CLASSES = len(class_names)
print(f"\nClasses ({NUM_CLASSES}): {class_names}")
assert NUM_CLASSES == 27, f"Expected 27 classes, got {NUM_CLASSES}"

with open(CLASSES_NAME, 'w') as f:
    json.dump(class_names, f)
print(f"Saved {CLASSES_NAME}")

# Phase 2 training set: webcam only
print(f"\nLoading Phase 2 (webcam-only) training set at {IMAGE_SIZE}...")
train_dataset_p2 = tf.keras.utils.image_dataset_from_directory(
    WEBCAM_ONLY_DIR,
    image_size=IMAGE_SIZE,
    batch_size=None,
    label_mode='categorical',
    shuffle=True,
    seed=SEED,
)

# Critical: class indices must match between phases. If they don't, Phase 2
# fine-tuning would update the wrong output neurons.
assert train_dataset_p2.class_names == class_names, (
    f"Class name mismatch between phases!\n"
    f"  Phase 1: {class_names}\n"
    f"  Phase 2: {train_dataset_p2.class_names}"
)

# Pipeline: shuffle BEFORE batching, then batch + prefetch
train_dataset_p1 = (
    train_dataset_p1
    .shuffle(20000, seed=SEED, reshuffle_each_iteration=True)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
train_dataset_p2 = (
    train_dataset_p2
    .shuffle(10000, seed=SEED, reshuffle_each_iteration=True)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
validation_dataset = (
    validation_dataset
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

steps_per_epoch_p1 = tf.data.experimental.cardinality(train_dataset_p1).numpy()
if steps_per_epoch_p1 <= 0:
    print("Phase 1 cardinality unknown; counting manually (one pass)...")
    steps_per_epoch_p1 = sum(1 for _ in train_dataset_p1)
print(f"Phase 1 steps per epoch: {steps_per_epoch_p1}")

steps_per_epoch_p2 = tf.data.experimental.cardinality(train_dataset_p2).numpy()
if steps_per_epoch_p2 <= 0:
    print("Phase 2 cardinality unknown; counting manually (one pass)...")
    steps_per_epoch_p2 = sum(1 for _ in train_dataset_p2)
print(f"Phase 2 steps per epoch: {steps_per_epoch_p2}")

# ══════════════════════════════════════════════════════════════════════════════
# Model architecture — MUST match live_asl_Final.py exactly (Bug 8)
# ══════════════════════════════════════════════════════════════════════════════

def build_augmentation():
    """RandomFlip intentionally absent — ASL signs are hand-specific."""
    return keras.Sequential([
        layers.RandomRotation(0.08),
        layers.RandomZoom(0.15),
        layers.RandomTranslation(0.10, 0.10),
        layers.RandomBrightness(0.4),
        layers.RandomContrast(0.4),
    ], name="augmentation")


def build_model(num_classes, image_size, dropout_rate=0.3):
    """Must produce the identical graph in train_asl_Final.py and
    live_asl_Final.py — see Bug 8."""
    aug = build_augmentation()

    base = keras.applications.MobileNetV2(
        input_shape=image_size + (3,),
        include_top=False,
        weights='imagenet',
    )
    base.trainable = False

    inputs = keras.Input(shape=image_size + (3,))
    x = aug(inputs)
    x = keras.applications.mobilenet_v2.preprocess_input(x)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(dropout_rate)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(num_classes, activation='softmax', dtype='float32')(x)
    return keras.Model(inputs, outputs), base


model, base_model = build_model(NUM_CLASSES, IMAGE_SIZE, DROPOUT_RATE)

# ══════════════════════════════════════════════════════════════════════════════
# Phase 1: frozen backbone, Kaggle + webcam
# ══════════════════════════════════════════════════════════════════════════════

phase1_schedule = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=INITIAL_LR_FROZEN,
    decay_steps=steps_per_epoch_p1 * EPOCHS_FROZEN,
    alpha=0.01,
)

model.compile(
    optimizer=keras.optimizers.AdamW(
        learning_rate=phase1_schedule,
        weight_decay=WEIGHT_DECAY,
    ),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics=['accuracy'],
)

model.summary()

print("\n" + "=" * 60)
print("PHASE 1: training classifier head on Kaggle + webcam")
print("=" * 60)

history = model.fit(
    train_dataset_p1,
    epochs=EPOCHS_FROZEN,
    validation_data=validation_dataset,
    callbacks=[
        keras.callbacks.EarlyStopping(
            patience=7, restore_best_weights=True,
            monitor='val_accuracy', verbose=1,
        ),
    ],
    verbose=1,
)

phase1_results = model.evaluate(validation_dataset, verbose=0)
print(f"\nPhase 1 — Val Loss: {phase1_results[0]:.4f} | "
      f"Val Accuracy: {phase1_results[1]:.4f}")

model.save_weights(PHASE1_BACKUP)
print(f"Phase 1 weights backed up to {PHASE1_BACKUP}.")

# ══════════════════════════════════════════════════════════════════════════════
# Phase 2: fine-tune top FINETUNE_LAYERS layers, WEBCAM ONLY
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 60)
print(f"PHASE 2: fine-tuning top {FINETUNE_LAYERS} backbone layers (webcam only)")
print("=" * 60)

base_model.trainable = True
for layer in base_model.layers[:-FINETUNE_LAYERS]:
    layer.trainable = False

# BN stays in inference mode (training=False was pinned in build_model).
trainable_count = sum(1 for l in base_model.layers if l.trainable)
frozen_count    = sum(1 for l in base_model.layers if not l.trainable)
print(f"Backbone: {trainable_count} trainable, {frozen_count} frozen")

phase2_schedule = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=INITIAL_LR_FINETUNE,
    decay_steps=steps_per_epoch_p2 * EPOCHS_FINETUNE,
    alpha=0.01,
)

model.compile(
    optimizer=keras.optimizers.AdamW(
        learning_rate=phase2_schedule,
        weight_decay=WEIGHT_DECAY,
    ),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics=['accuracy'],
)

history_finetune = model.fit(
    train_dataset_p2,
    epochs=EPOCHS_FINETUNE,
    validation_data=validation_dataset,
    callbacks=[
        keras.callbacks.EarlyStopping(
            patience=5, restore_best_weights=True,
            monitor='val_accuracy', verbose=1,
        ),
    ],
    verbose=1,
)

final_results = model.evaluate(validation_dataset, verbose=0)
print(f"\nFinal Val Loss: {final_results[0]:.4f} | "
      f"Final Val Accuracy: {final_results[1]:.4f}")

# ══════════════════════════════════════════════════════════════════════════════
# Confusion matrix & per-class diagnostics
# ══════════════════════════════════════════════════════════════════════════════

all_predictions = []
all_labels = []
for images, labels in validation_dataset:
    preds = model.predict(images, verbose=0)
    all_predictions.extend(np.argmax(preds, axis=1))
    all_labels.extend(np.argmax(labels.numpy(), axis=1))

all_predictions = np.array(all_predictions)
all_labels      = np.array(all_labels)

print(f"\n{'':>8}", end="")
for name in class_names:
    print(f"{name:>5}", end="")
print("  <- Predicted")
print("-" * (8 + 5 * len(class_names) + 15))

for i, true_name in enumerate(class_names):
    print(f"{true_name:>6} |", end="")
    for j in range(len(class_names)):
        count = np.sum((all_labels == i) & (all_predictions == j))
        if i == j:
            print(f" [{count:>2}]", end="")
        elif count > 0:
            print(f"  {count:>2}", end="")
        else:
            print(f"    .", end="")
    print(f"  | {true_name}")

class_accs = []
for i, name in enumerate(class_names):
    mask = all_labels == i
    if mask.sum() > 0:
        acc     = np.sum(all_predictions[mask] == i) / mask.sum()
        total   = int(mask.sum())
        correct = int(np.sum(all_predictions[mask] == i))
        class_accs.append((name, acc, correct, total))

class_accs.sort(key=lambda x: x[1])
print("\nPer-class accuracy (worst first):")
for name, acc, correct, total in class_accs:
    bar = "#" * int(acc * 20) + "." * (20 - int(acc * 20))
    print(f"  {name}: [{bar}] {acc:.1%} ({correct}/{total})")

overall_acc = np.sum(all_predictions == all_labels) / len(all_labels)
print(f"\nOverall accuracy: {overall_acc:.1%}")

print("\nTop confusions:")
confusions = []
for i in range(len(class_names)):
    for j in range(len(class_names)):
        if i != j:
            count = int(np.sum((all_labels == i) & (all_predictions == j)))
            if count > 0:
                confusions.append((class_names[i], class_names[j], count))
confusions.sort(key=lambda x: x[2], reverse=True)
for true_cls, pred_cls, count in confusions[:20]:
    print(f"  {true_cls} predicted as {pred_cls}: {count} times")

weak_letters = [name for name, acc, _, _ in class_accs if acc < 0.90]
print("\nLetters below 90% accuracy (focus future data capture here):")
if weak_letters:
    print(f"  {', '.join(weak_letters)}")
else:
    print("  None — all classes above 90%!")

# ══════════════════════════════════════════════════════════════════════════════
# Save final weights
# ══════════════════════════════════════════════════════════════════════════════

model.save_weights(WEIGHTS_NAME)
print(f"\nSaved {WEIGHTS_NAME}")
print(f"Saved {CLASSES_NAME}")

print("\nDownload these two files to your Mac and place them next to live_asl_Final.py:")
print(f"  - {WEIGHTS_NAME}")
print(f"  - {CLASSES_NAME}")

Unzip done.
Mixed precision enabled: training in float16
Kaggle dataset found at: asl_alphabet_train/asl_alphabet_train
Removed 'nothing' class
Removed 'del' class

Building webcam-only pool for Phase 2 fine-tuning...
Webcam-only pool: 82320 images across 27 class folders

Merging webcam_training_data_Final into Kaggle dataset (for Phase 1)...
  A: added 3028 images
  B: added 3005 images
  C: added 3013 images
  D: added 3094 images
  E: added 3102 images
  F: added 3086 images
  G: added 3077 images
  H: added 3085 images
  I: added 3075 images
  J: added 3054 images
  K: added 3054 images
  L: added 3031 images
  M: added 3070 images
  N: added 3057 images
  O: added 3046 images
  P: added 3018 images
  Q: added 3039 images
  R: added 3061 images
  S: added 3046 images
  T: added 3073 images
  U: added 3048 images
  V: added 3046 images
  W: added 3035 images
  X: added 3500 images
  Y: added 3063 images
  Z: added 3046 images
  space: added 2468 images
  Subtotal from webcam_traini

/tmp/ipykernel_696/2003816057.py:336: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base = keras.applications.MobileNetV2(


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ augmentation (Sequential)       │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 8, 8, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       655,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 27)             │         6,939 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,052,123 (11.64 MB)

 Trainable params: 794,139 (3.03 MB)

 Non-trainable params: 2,257,984 (8.61 MB)


PHASE 1: training classifier head on Kaggle + webcam
Epoch 1/30
5104/5104 ━━━━━━━━━━━━━━━━━━━━ 206s 35ms/step - accuracy: 0.7039 - loss: 1.5096 - val_accuracy: 0.7991 - val_loss: 1.2608
Epoch 2/30
5104/5104 ━━━━━━━━━━━━━━━━━━━━ 153s 29ms/step - accuracy: 0.8108 - loss: 1.2342 - val_accuracy: 0.8275 - val_loss: 1.1386
Epoch 3/30
5104/5104 ━━━━━━━━━━━━━━━━━━━━ 153s 29ms/step - accuracy: 0.8361 - loss: 1.1588 - val_accuracy: 0.8073 - val_loss: 1.1907
Epoch 4/30
5104/5104 ━━━━━━━━━━━━━━━━━━━━ 154s 30ms/step - accuracy: 0.8488 - loss: 1.1145 - val_accuracy: 0.8184 - val_loss: 1.1369
Epoch 5/30
5104/5104 ━━━━━━━━━━━━━━━━━━━━ 152s 29ms/step - accuracy: 0.8573 - loss: 1.0875 - val_accuracy: 0.8419 - val_loss: 1.0961
Epoch 6/30
5104/5104 ━━━━━━━━━━━━━━━━━━━━ 154s 30ms/step - accuracy: 0.8655 - loss: 1.0660 - val_accuracy: 0.8211 - val_loss: 1.1197
Epoch 7/30
5104/5104 ━━━━━━━━━━━━━━━━━━━━ 153s 29ms/step - accuracy: 0.8722 - loss: 1.0453 - val_accuracy: 0.8236 - val_loss: 1.1448
Epoch 8/30
5104